# Train on Kaggle GPU

Edit this notebook **locally** (VS Code). Run it on a **Kaggle GPU** with:

```powershell
python -m kgpu run
```

The same notebook also runs locally on CPU as a smoke test — it auto-detects
the environment and shrinks the workload when there is no GPU, so you can
catch syntax and shape errors before spending GPU quota.

Anything written to `OUT_DIR` is downloaded back to `results/` after the run.

## 1. Environment + hyperparameters

In [ ]:
import json
import os
import platform
import time
from pathlib import Path

import torch

# Kaggle mounts /kaggle/working as the only writable, persisted directory.
# Locally we write elsewhere: results/ is reserved for Kaggle downloads and is
# wiped on every `python -m kgpu pull`.
ON_KAGGLE = Path("/kaggle/working").exists()
OUT_DIR = Path("/kaggle/working/results") if ON_KAGGLE else Path("../local_runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE = DEVICE.type != "cuda"  # no GPU -> tiny run, just prove the code executes

SEED = 1337
EPOCHS = 3 if SMOKE else 30
BATCH_SIZE = 128 if SMOKE else 1024
N_TRAIN = 2_000 if SMOKE else 200_000
N_VAL = 1_000 if SMOKE else 20_000
N_FEATURES = 64
N_CLASSES = 10
LR = 3e-3
USE_AMP = DEVICE.type == "cuda"

torch.manual_seed(SEED)

print(f"on kaggle : {ON_KAGGLE}")
print(f"python    : {platform.python_version()}")
print(f"torch     : {torch.__version__}  (cuda build: {torch.version.cuda})")
print(f"device    : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"gpu       : {torch.cuda.get_device_name(0)} x{torch.cuda.device_count()}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"gpu memory: {mem:.1f} GiB")
else:
    print("gpu       : none -> SMOKE MODE (shrunken workload)")
print(f"out dir   : {OUT_DIR.resolve()}")

# Kaggle's default image ships a PyTorch with no kernels for older cards (the
# P100 is sm_60). Without this check the run dies mid-training with a bare
# "CUDA error: no kernel image is available for execution on the device".
if DEVICE.type == "cuda":
    major, minor = torch.cuda.get_device_capability(0)
    arch = f"sm_{major}{minor}"
    supported = torch.cuda.get_arch_list()
    if arch not in supported:
        raise RuntimeError(
            f"{torch.cuda.get_device_name(0)} is {arch}, but this PyTorch build "
            f"only has kernels for {', '.join(a for a in supported if a.startswith('sm_'))}.\n"
            f"Set \"accelerator\": \"NvidiaTeslaT4\" in kaggle_config.json and re-run."
        )
    print(f"arch      : {arch} (supported by this torch build)")

## 2. Data

A synthetic, linearly-mixed classification problem so the notebook has zero
external dependencies and is reproducible. Swap this cell for your real
dataset — on Kaggle, attached datasets appear under `/kaggle/input/<slug>/`
(add them via `dataset_sources` in `kaggle_config.json`).

In [ ]:
from torch.utils.data import DataLoader, TensorDataset


def make_dataset(n: int, generator: torch.Generator) -> TensorDataset:
    """Deterministic synthetic classification data."""
    x = torch.randn(n, N_FEATURES, generator=generator)
    # Fixed random projection -> class logits, plus label noise.
    w = torch.randn(N_FEATURES, N_CLASSES, generator=torch.Generator().manual_seed(SEED))
    logits = x @ w + 0.35 * torch.randn(n, N_CLASSES, generator=generator)
    y = logits.argmax(dim=1)
    return TensorDataset(x, y)


g = torch.Generator().manual_seed(SEED)
train_ds = make_dataset(N_TRAIN, g)
val_ds = make_dataset(N_VAL, g)

# num_workers=2 matches Kaggle's 2 vCPU-per-GPU shape; pin_memory only helps on CUDA.
common = dict(num_workers=2 if ON_KAGGLE else 0, pin_memory=DEVICE.type == "cuda")
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **common)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **common)

print(f"train: {len(train_ds):,} samples / {len(train_dl):,} batches")
print(f"val  : {len(val_ds):,} samples / {len(val_dl):,} batches")

## 3. Model

In [ ]:
import torch.nn as nn


class MLP(nn.Module):
    def __init__(self, in_features: int, n_classes: int, width: int = 512, depth: int = 4):
        super().__init__()
        layers: list[nn.Module] = []
        d = in_features
        for _ in range(depth):
            layers += [nn.Linear(d, width), nn.BatchNorm1d(width), nn.GELU(), nn.Dropout(0.1)]
            d = width
        layers.append(nn.Linear(d, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


model = MLP(N_FEATURES, N_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_dl)
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

print(model)
print(f"\nparameters: {n_params:,}")

## 4. Train

In [ ]:
@torch.no_grad()
def evaluate() -> tuple[float, float]:
    model.eval()
    total_loss, correct, seen = 0.0, 0, 0
    for xb, yb in val_dl:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            out = model(xb)
            loss = criterion(out, yb)
        total_loss += loss.item() * yb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        seen += yb.size(0)
    return total_loss / seen, correct / seen


history = []
best_acc = 0.0
start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_start = time.perf_counter()
    running, seen = 0.0, 0

    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            loss = criterion(model(xb), yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += loss.item() * yb.size(0)
        seen += yb.size(0)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    train_loss = running / seen
    val_loss, val_acc = evaluate()
    secs = time.perf_counter() - epoch_start
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "seconds": secs,
        }
    )

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), OUT_DIR / "model_best.pt")

    # flush=True so logs stream out of the Kaggle worker as the run progresses.
    print(
        f"epoch {epoch:>3}/{EPOCHS} | train {train_loss:.4f} | "
        f"val {val_loss:.4f} | acc {val_acc:.4f} | {secs:5.2f}s",
        flush=True,
    )

total_seconds = time.perf_counter() - start
print(f"\ntrained {EPOCHS} epochs in {total_seconds:.1f}s | best val acc {best_acc:.4f}")

## 5. Save artifacts

Everything under `OUT_DIR` comes back to your machine in `results/`.

In [ ]:
torch.save(model.state_dict(), OUT_DIR / "model_final.pt")

summary = {
    "smoke_mode": SMOKE,
    "on_kaggle": ON_KAGGLE,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "torch": torch.__version__,
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "n_train": N_TRAIN,
    "n_params": n_params,
    "amp": USE_AMP,
    "best_val_acc": best_acc,
    "final_val_acc": history[-1]["val_acc"],
    "total_seconds": total_seconds,
    "history": history,
}

(OUT_DIR / "metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps({k: v for k, v in summary.items() if k != "history"}, indent=2))
print("\nartifacts:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:<20} {p.stat().st_size / 1024:>10.1f} KiB")